# Notebook 02 — Facial-Cue Analysis & Honest Evaluation on DAIC-WOZ / E-DAIC

**Input:** the two tables written by Notebook 01 —
`outputs/prep/daic_subject_features.csv` and `outputs/prep/edaic_subject_features.csv`.
Run Notebook 01 first, in the same folder.

### What this notebook answers
1. **Which facial cues relate to depression severity?** (correlations, FDR-corrected)
2. **Is there a real signal, or is it chance?** (permutation test on AUC)
3. **How well can it be predicted, measured honestly?** Leave-one-subject-out (LOSO) with
   feature selection and tuning *inside* each fold — so nothing about the test person leaks
   into training — plus bootstrap 95% CIs and a comparison to trivial baselines.
4. **Does it hold on E-DAIC's official test split?**
5. **Does it transfer across corpora?** (train DAIC → test E-DAIC and back, on shared AUs)
6. **Which cue families matter most?** (grouped permutation importance: AU vs gaze vs pose,
   and per Action Unit)

Needs only `numpy, pandas, scipy, scikit-learn, matplotlib`. No GPU.

## 1. Config

In [ ]:
import os, re, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, f_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, KFold, LeaveOneOut,
                                     RepeatedStratifiedKFold, cross_val_predict,
                                     permutation_test_score)
from sklearn.base import clone
from sklearn.metrics import (f1_score, balanced_accuracy_score, roc_auc_score,
                             mean_squared_error, mean_absolute_error)

PREP_DIR  = "outputs/prep"
DAIC_CSV  = os.path.join(PREP_DIR, "daic_subject_features.csv")
EDAIC_CSV = os.path.join(PREP_DIR, "edaic_subject_features.csv")
OUT       = "outputs/analysis"; os.makedirs(OUT, exist_ok=True)

OUTER     = "loso"   # "loso" (leave-one-subject-out, publishable) or "rskf" (faster, 5x2 repeated)
N_PERM    = 200      # permutations for the chance test (500+ for the final paper)
N_BOOT    = 2000     # bootstrap resamples for 95% CIs
SEED      = 42
N_JOBS    = -1       # use all CPU cores

# chart styling (static figures for the paper)
INK, INK2, GRID, BAR, NEG = "#0b0b0b", "#52514e", "#e6e5e0", "#2a78d6", "#eb6834"
plt.rcParams.update({"axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10, "figure.dpi": 110})
print("DAIC  csv:", os.path.exists(DAIC_CSV), "| E-DAIC csv:", os.path.exists(EDAIC_CSV))

## 2. Load the tables and label every feature with its facial cue

Column names look like `au:AU12_r:mean` or `pose:Rx:std` or (E-DAIC) `au:gaze_0_x:p75`. We map
each to a **cue family** (AU / gaze / pose) and, for AUs, to its **Action Unit** so the analysis
can talk about AU12, AU15 … rather than 300 anonymous columns.

In [ ]:
META = ["pid", "split", "gender", "phq_binary", "phq_score"]
AU_NAMES = {"AU01":"inner brow raiser","AU02":"outer brow raiser","AU04":"brow lowerer",
            "AU05":"upper lid raiser","AU06":"cheek raiser","AU07":"lid tightener",
            "AU09":"nose wrinkler","AU10":"upper lip raiser","AU12":"lip corner puller (smile)",
            "AU14":"dimpler","AU15":"lip corner depressor","AU17":"chin raiser",
            "AU20":"lip stretcher","AU23":"lip tightener","AU25":"lips part",
            "AU26":"jaw drop","AU28":"lip suck","AU45":"blink"}

def cue_family(col):
    parts = col.split(":")
    if len(parts) < 3: return "other"
    prefix, name = parts[0], parts[1]
    if re.match(r"AU\d+_[rc]$", name):                  return "AU"
    if prefix == "gaze" or name.startswith("gaze"):     return "gaze"
    if prefix == "pose" or name.startswith("pose"):     return "pose"
    return "other"

def au_id(col):
    m = re.search(r"(AU\d+)_[rc]", col); return m.group(1) if m else None

def load(path, tag):
    if not os.path.exists(path):
        print(f"{tag}: file not found — run Notebook 01 first"); return None
    df = pd.read_csv(path)
    df = df[df["phq_binary"].notna()].copy()
    df["phq_binary"] = df["phq_binary"].astype(int)
    feats = [c for c in df.columns if c not in META]
    fam = pd.Series({c: cue_family(c) for c in feats})
    print(f"{tag}: {len(df)} labelled subjects | {len(feats)} features "
          f"{dict(fam.value_counts())} | depressed {df.phq_binary.sum()} "
          f"({100*df.phq_binary.mean():.0f}%)")
    assert df["pid"].is_unique, "duplicate participants — re-run the fixed Notebook 0"
    return {"tag": tag, "df": df, "feats": feats, "fam": fam}

DATA = {k: v for k, v in [("DAIC", load(DAIC_CSV, "DAIC")),
                          ("E-DAIC", load(EDAIC_CSV, "E-DAIC"))] if v is not None}

## 3. Which cues correlate with depression severity?

Spearman ρ between each feature and the PHQ-8 score across all labelled subjects, with
**Benjamini–Hochberg FDR** correction for the hundreds of tests. This is *descriptive* (it uses
every subject); prediction is measured separately and honestly in §5.

In [ ]:
def bh_fdr(p):
    p = np.asarray(p); n = len(p); order = np.argsort(p)
    q = np.empty(n); prev = 1.0
    for rank, i in enumerate(order[::-1], 1):
        idx = n - rank + 1
        prev = min(prev, p[i] * n / idx); q[i] = prev
    return q

CORR = {}
for tag, D in DATA.items():
    df = D["df"]; y = df["phq_score"].values
    rows = []
    for c in D["feats"]:
        x = df[c].values; ok = np.isfinite(x) & np.isfinite(y)
        if ok.sum() < 10 or np.nanstd(x[ok]) == 0: continue
        rho, p = stats.spearmanr(x[ok], y[ok])
        rows.append((c, D["fam"][c], au_id(c), rho, p))
    r = pd.DataFrame(rows, columns=["feature", "family", "au", "rho", "p"])
    r["q_fdr"] = bh_fdr(r["p"].values); r["abs_rho"] = r.rho.abs()
    r = r.sort_values("abs_rho", ascending=False)
    CORR[tag] = r
    r.to_csv(os.path.join(OUT, f"{tag}_correlations.csv"), index=False)
    print(f"\n{tag}: {int((r.q_fdr < 0.05).sum())} features significant after FDR (q<0.05) "
          f"out of {len(r)}")
    print(r.head(12)[["feature", "family", "rho", "p", "q_fdr"]].to_string(index=False))

In [ ]:
def plot_top_corr(r, tag, n=15):
    top = r.head(n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 0.32 * n + 1))
    colors = [BAR if v > 0 else NEG for v in top.rho]
    ax.barh(range(len(top)), top.rho, color=colors, height=0.6)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([f.replace("au:", "").replace("gaze:", "gaze ").replace("pose:", "pose ")
                        for f in top.feature], fontsize=8)
    ax.axvline(0, color=INK2, lw=0.8); ax.grid(axis="x", color=GRID, lw=0.6); ax.set_axisbelow(True)
    ax.set_xlabel("Spearman ρ with PHQ-8 score  (blue = higher with depression, orange = lower)")
    ax.set_title(f"{tag}: features most associated with depression severity", color=INK, loc="left")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, f"fig_{tag}_top_correlations.png"), dpi=200)
    plt.show()

for tag, r in CORR.items(): plot_top_corr(r, tag)

## 4. Evaluation machinery (leakage-safe by construction)

Every model is a single pipeline — impute → drop constants → standardise → **select k best
features** → classifier — and the *whole pipeline* is tuned inside each training fold
(`GridSearchCV`). Feature selection on all data before cross-validation is the most common
silent leak in this literature; here it cannot happen.

The outer loop is **leave-one-subject-out**: each person is predicted by a model that never saw
them. Pooled predictions give macro-F1 / balanced accuracy / AUC (binary) and RMSE / MAE / CCC
(severity), each with a bootstrap 95% CI, next to a **trivial baseline** (majority class /
mean PHQ) evaluated the same way.

In [ ]:
def ccc(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    vy, vp = y.var(), p.var()
    cov = np.mean((y - y.mean()) * (p - p.mean()))
    return float(2 * cov / (vy + vp + (y.mean() - p.mean()) ** 2 + 1e-12))

def k_grid(n_feat):
    return sorted({k for k in (10, 25, 50, 100) if k < n_feat} | {"all"}, key=str)

def clf_search(n_feat):
    pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("var", VarianceThreshold()),
                     ("sc", StandardScaler()), ("sel", SelectKBest(f_classif)),
                     ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))])
    grid = {"sel__k": k_grid(n_feat), "clf__C": [0.01, 0.1, 1.0]}
    return GridSearchCV(pipe, grid, scoring="roc_auc",
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=N_JOBS)

def reg_search(n_feat):
    pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("var", VarianceThreshold()),
                     ("sc", StandardScaler()), ("sel", SelectKBest(f_regression)),
                     ("reg", Ridge())])
    grid = {"sel__k": k_grid(n_feat), "reg__alpha": [1, 10, 100, 1000]}
    return GridSearchCV(pipe, grid, scoring="neg_root_mean_squared_error",
                        cv=KFold(5, shuffle=True, random_state=SEED), n_jobs=N_JOBS)

def outer_cv(y_strat):
    if OUTER == "loso": return LeaveOneOut()
    return RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)

def pooled_predict(est, X, y, cv, method):
    if OUTER == "loso":
        return cross_val_predict(est, X, y, cv=cv, method=method)
    # repeated k-fold: average predictions across repeats
    preds, counts = np.zeros(len(y)), np.zeros(len(y))
    for tr, te in cv.split(X, (y > np.median(y)).astype(int) if method == "predict" else y):
        e = clone(est).fit(X[tr], y[tr])
        p = e.predict_proba(X[te])[:, 1] if method == "predict_proba" else e.predict(X[te])
        preds[te] += p; counts[te] += 1
    return preds / np.maximum(counts, 1)

def boot_ci(fn, y, p, n=N_BOOT):
    rng = np.random.default_rng(SEED); vals = []
    y, p = np.asarray(y), np.asarray(p)
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2 and fn is not None and fn.__name__ in ("auc_", "f1_", "bacc_"):
            continue
        try: vals.append(fn(y[i], p[i]))
        except Exception: pass
    return np.percentile(vals, [2.5, 97.5]) if vals else (np.nan, np.nan)

def auc_(y, p):  return roc_auc_score(y, p)
def f1_(y, p):   return f1_score(y, (p >= 0.5).astype(int), average="macro")
def bacc_(y, p): return balanced_accuracy_score(y, (p >= 0.5).astype(int))
def rmse_(y, p): return mean_squared_error(y, p) ** 0.5
def mae_(y, p):  return mean_absolute_error(y, p)
def ccc_(y, p):  return ccc(y, p)

def fmt(fn, y, p):
    v = fn(y, p); lo, hi = boot_ci(fn, y, p)
    return f"{v:.3f} [{lo:.3f}, {hi:.3f}]", v
print("evaluation machinery ready | outer loop:", OUTER)

## 5. Leave-one-subject-out results (binary + severity)

This is the main results table for the paper. Expect it to take a few minutes per dataset
(each subject's model is tuned separately). Compare every row to its **baseline** row: a model
only earns its place if its CI clears the baseline.

In [ ]:
RESULTS = []
PRED = {}
for tag, D in DATA.items():
    df, feats = D["df"], D["feats"]
    X = df[feats].values.astype(float); yb = df["phq_binary"].values; ys = df["phq_score"].values.astype(float)
    t0 = time.time()
    # --- binary ---
    p_model = pooled_predict(clf_search(len(feats)), X, yb, outer_cv(yb), "predict_proba")
    p_model = p_model[:, 1] if p_model.ndim == 2 else p_model
    p_base  = pooled_predict(DummyClassifier(strategy="prior"), X, yb, outer_cv(yb), "predict_proba")
    p_base  = p_base[:, 1] if p_base.ndim == 2 else p_base
    PRED[tag] = {"yb": yb, "p": p_model}
    for name, p in [("logistic + feature selection", p_model), ("baseline (class prior)", p_base)]:
        f1s, _ = fmt(f1_, yb, p); bas, _ = fmt(bacc_, yb, p)
        # a trivial baseline is uninformative: AUC = 0.5 by definition (LOSO makes the
        # prior drift against the held-out label, which would otherwise show a fake 0.0)
        aus = "0.500 (by definition)" if name.startswith("baseline") else fmt(auc_, yb, p)[0]
        RESULTS.append(dict(dataset=tag, task="binary", model=name,
                            macro_F1=f1s, bal_acc=bas, AUC=aus))
    # --- severity ---
    okm = np.isfinite(ys); Xs, yss = X[okm], ys[okm]
    r_model = pooled_predict(reg_search(len(feats)), Xs, yss, outer_cv(yss), "predict")
    r_base  = pooled_predict(DummyRegressor(strategy="mean"), Xs, yss, outer_cv(yss), "predict")
    for name, p in [("ridge + feature selection", r_model), ("baseline (mean PHQ)", r_base)]:
        RESULTS.append(dict(dataset=tag, task="severity", model=name,
                            RMSE=fmt(rmse_, yss, p)[0], MAE=fmt(mae_, yss, p)[0],
                            CCC="0.000 (by definition)" if name.startswith("baseline") else fmt(ccc_, yss, p)[0]))
    print(f"{tag} done in {time.time()-t0:.0f}s")

res = pd.DataFrame(RESULTS)
res.to_csv(os.path.join(OUT, f"results_{OUTER}.csv"), index=False)
for task in ["binary", "severity"]:
    cols = ["dataset", "model"] + (["macro_F1", "bal_acc", "AUC"] if task == "binary" else ["RMSE", "MAE", "CCC"])
    print(f"\n==== {task.upper()} — {OUTER.upper()} (value [95% bootstrap CI]) ====")
    print(res[res.task == task][cols].to_string(index=False))

## 6. Is the signal real? — permutation test

Shuffles the depression labels `N_PERM` times and re-runs the same cross-validated model.
If the real AUC is not clearly above the shuffled-label distribution, **there is no evidence
of a signal** — and that must be reported as such. A fixed, simple pipeline is used here
(k=25 features, C=0.1) so the test is fast and not itself tuned on the result.

In [ ]:
PERM = {}
for tag, D in DATA.items():
    df, feats = D["df"], D["feats"]
    X = df[feats].values.astype(float); yb = df["phq_binary"].values
    k = 25 if len(feats) > 25 else "all"
    pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("var", VarianceThreshold()),
                     ("sc", StandardScaler()), ("sel", SelectKBest(f_classif, k=k)),
                     ("clf", LogisticRegression(C=0.1, max_iter=5000, class_weight="balanced"))])
    score, perm_scores, pval = permutation_test_score(
        pipe, X, yb, scoring="roc_auc", cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
        n_permutations=N_PERM, random_state=SEED, n_jobs=N_JOBS)
    PERM[tag] = dict(auc=float(score), p=float(pval), null_mean=float(perm_scores.mean()))
    verdict = "SIGNAL above chance" if pval < 0.05 else "NOT distinguishable from chance"
    print(f"{tag}: AUC={score:.3f} | shuffled-label AUC mean={perm_scores.mean():.3f} | "
          f"p={pval:.4f}  ->  {verdict}")
    fig, ax = plt.subplots(figsize=(6, 2.6))
    ax.hist(perm_scores, bins=25, color=GRID, edgecolor=INK2, lw=0.4)
    ax.axvline(score, color=BAR, lw=2); ax.text(score, ax.get_ylim()[1]*0.9, f"  real AUC {score:.3f}", color=INK)
    ax.set_xlabel("AUC with shuffled labels"); ax.set_ylabel("count")
    ax.set_title(f"{tag}: permutation test (p = {pval:.3f})", color=INK, loc="left")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, f"fig_{tag}_permutation.png"), dpi=200); plt.show()

## 7. E-DAIC official split (train + dev → test)

The comparable-to-literature number: tune on train+dev only, predict the held-out official
test set once.

In [ ]:
if "E-DAIC" in DATA:
    D = DATA["E-DAIC"]; df, feats = D["df"], D["feats"]
    tr = df[df.split.isin(["train", "dev"])]; te = df[df.split == "test"]
    if len(te):
        Xtr, Xte = tr[feats].values.astype(float), te[feats].values.astype(float)
        m = clf_search(len(feats)).fit(Xtr, tr.phq_binary.values)
        p = m.predict_proba(Xte)[:, 1]; yte = te.phq_binary.values
        print("best params:", m.best_params_)
        print("binary  macro-F1", fmt(f1_, yte, p)[0], "| bal-acc", fmt(bacc_, yte, p)[0],
              "| AUC", fmt(auc_, yte, p)[0])
        trs, tes = tr[tr.phq_score.notna()], te[te.phq_score.notna()]
        r = reg_search(len(feats)).fit(trs[feats].values.astype(float), trs.phq_score.values)
        pr = r.predict(tes[feats].values.astype(float)); yr = tes.phq_score.values
        base = np.full(len(yr), trs.phq_score.mean())
        print("severity RMSE", fmt(rmse_, yr, pr)[0], "| MAE", fmt(mae_, yr, pr)[0],
              "| CCC", fmt(ccc_, yr, pr)[0])
        print("   mean-PHQ baseline RMSE", f"{rmse_(yr, base):.3f}", "| MAE", f"{mae_(yr, base):.3f}")
    else:
        print("no labelled E-DAIC test split")

## 8. Cross-corpus transfer on shared Action Units

DAIC uses CLNF, E-DAIC uses OpenFace 2.1, so only the **AU** features share names. Each corpus's
features are z-scored *within that corpus* (label-free, so no leakage) to absorb the different
toolkits' scales, then we train on one corpus and test on the other. Transfer is a strong
robustness result if it holds — and an honest limitation if it doesn't.

In [ ]:
if len(DATA) == 2:
    A, B = DATA["DAIC"], DATA["E-DAIC"]
    shared = sorted(set(c for c in A["feats"] if A["fam"][c] == "AU") &
                    set(c for c in B["feats"] if B["fam"][c] == "AU"))
    print(f"shared AU features: {len(shared)}")
    if len(shared) >= 5:
        def zs(df):
            X = SimpleImputer(strategy="median").fit_transform(df[shared].values.astype(float))
            return (X - X.mean(0)) / (X.std(0) + 1e-9)
        for (src, dst) in [(A, B), (B, A)]:
            Xs, Xd = zs(src["df"]), zs(dst["df"])
            m = clf_search(len(shared)).fit(Xs, src["df"].phq_binary.values)
            p = m.predict_proba(Xd)[:, 1]; yd = dst["df"].phq_binary.values
            print(f"train {src['tag']:6s} -> test {dst['tag']:6s}: AUC {fmt(auc_, yd, p)[0]} | "
                  f"macro-F1 {fmt(f1_, yd, p)[0]}")
    else:
        print("too few shared AU features for transfer")

## 9. Which cue families and Action Units carry the signal?

**Grouped permutation importance:** inside repeated cross-validation, shuffle *all* columns of
one group together in the held-out fold and measure how much AUC drops. Grouping matters —
permuting one of 300 correlated columns at a time underestimates everything. Groups:
(a) cue family (AU / gaze / pose), (b) each individual Action Unit.

In [ ]:
def grouped_importance(D, groups, n_rep=3):
    df, feats = D["df"], D["feats"]
    X = df[feats].values.astype(float); y = df["phq_binary"].values
    k = 25 if len(feats) > 25 else "all"
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=n_rep, random_state=SEED)
    rng = np.random.default_rng(SEED); drops = {g: [] for g in groups}
    for tr, te in rskf.split(X, y):
        pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("var", VarianceThreshold()),
                         ("sc", StandardScaler()), ("sel", SelectKBest(f_classif, k=k)),
                         ("clf", LogisticRegression(C=0.1, max_iter=5000, class_weight="balanced"))])
        pipe.fit(X[tr], y[tr])
        if len(np.unique(y[te])) < 2: continue
        base = roc_auc_score(y[te], pipe.predict_proba(X[te])[:, 1])
        for g, idx in groups.items():
            Xp = X[te].copy(); perm = rng.permutation(len(te)); Xp[:, idx] = Xp[perm][:, idx]
            drops[g].append(base - roc_auc_score(y[te], pipe.predict_proba(Xp)[:, 1]))
    out = pd.DataFrame({"group": list(drops),
                        "auc_drop_mean": [np.mean(v) for v in drops.values()],
                        "auc_drop_sd": [np.std(v) for v in drops.values()]})
    return out.sort_values("auc_drop_mean", ascending=False)

IMP = {}
for tag, D in DATA.items():
    feats = D["feats"]; idx = {f: i for i, f in enumerate(feats)}
    fam_groups = {f: [idx[c] for c in feats if D["fam"][c] == f] for f in D["fam"].unique() if f != "other"}
    au_groups = {}
    for c in feats:
        a = au_id(c)
        if a: au_groups.setdefault(a, []).append(idx[c])
    fam_imp = grouped_importance(D, fam_groups)
    au_imp  = grouped_importance(D, au_groups)
    au_imp["name"] = au_imp.group.map(AU_NAMES).fillna("")
    IMP[tag] = (fam_imp, au_imp)
    fam_imp.to_csv(os.path.join(OUT, f"{tag}_family_importance.csv"), index=False)
    au_imp.to_csv(os.path.join(OUT, f"{tag}_au_importance.csv"), index=False)
    print(f"\n{tag} — cue-family importance (AUC drop when the family is shuffled):")
    print(fam_imp.to_string(index=False))
    print(f"{tag} — top Action Units:")
    print(au_imp.head(8).to_string(index=False))

In [ ]:
for tag, (fam_imp, au_imp) in IMP.items():
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1, 1.6]})
    for ax, t, labels, title in [
        (axes[0], fam_imp, fam_imp.group, "cue family"),
        (axes[1], au_imp.head(10), [f"{g} {AU_NAMES.get(g,'')}" for g in au_imp.head(10).group], "Action Unit (top 10)")]:
        t = t.iloc[::-1]; labels = list(labels)[::-1]
        ax.barh(range(len(t)), t.auc_drop_mean, xerr=t.auc_drop_sd, color=BAR, height=0.6,
                error_kw={"ecolor": INK2, "lw": 0.8, "capsize": 2})
        ax.set_yticks(range(len(t))); ax.set_yticklabels(labels, fontsize=8)
        ax.axvline(0, color=INK2, lw=0.8); ax.grid(axis="x", color=GRID, lw=0.6); ax.set_axisbelow(True)
        ax.set_xlabel("AUC drop when shuffled"); ax.set_title(title, color=INK, loc="left", fontsize=10)
    fig.suptitle(f"{tag}: which facial cues the model relies on", x=0.01, ha="left", color=INK)
    plt.tight_layout(); plt.savefig(os.path.join(OUT, f"fig_{tag}_importance.png"), dpi=200); plt.show()

## 10. Summary for the paper

In [ ]:
summary = {"outer_cv": OUTER, "n_perm": N_PERM, "permutation_test": PERM,
           "n_subjects": {t: int(len(D["df"])) for t, D in DATA.items()},
           "n_significant_correlations_fdr05": {t: int((r.q_fdr < 0.05).sum()) for t, r in CORR.items()}}
with open(os.path.join(OUT, "summary.json"), "w") as f: json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print("\nAll tables and figures saved to", OUT)

## How to read and report these results

| If you see… | It means… | Write in the paper… |
|---|---|---|
| Permutation p < 0.05 and LOSO AUC CI above 0.5 | A real but possibly modest signal | Report AUC with CI and p; the cue analysis (§3, §9) is the contribution |
| Permutation p ≥ 0.05 | No evidence of signal from *session-averaged* facial features | Report it honestly; it motivates temporal modelling (the next notebook) |
| Severity model not better than mean-PHQ baseline | Averages can't predict severity | Report the baseline alongside — reviewers check this |
| Cross-corpus AUC ≈ 0.5 | Cues don't transfer CLNF ↔ OpenFace | A limitation + motivation for domain-robust features |

**Never** report the random 5-fold numbers from Notebook 01 as results — they were a sanity check.

### Outputs → manuscript
| Section | File |
|---|---|
| Table: cue–severity correlations | `*_correlations.csv`, `fig_*_top_correlations.png` |
| Table: LOSO results vs baselines (with CIs) | `results_loso.csv` |
| Figure: permutation test | `fig_*_permutation.png` |
| Official-split + cross-corpus results | §7, §8 output |
| Figure: cue-family and AU importance | `fig_*_importance.png`, `*_importance.csv` |